# 🤖 CodeAlpha — AI Chatbot (AlphaBot)
### Google Colab Edition | Powered by Anthropic Claude

---
**▶️ HOW TO RUN:**
1. Run **Cell 1** → installs packages
2. Run **Cell 2** → enter your API key
3. Run **Cell 3** → chat UI appears below!
4. Run **Cell 4** (optional) → save/download chat log

> 🔑 Get a free API key at: https://console.anthropic.com
---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 ── Install Required Packages
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

print('📦 Installing required packages...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'anthropic', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ anthropic installed successfully!')
else:
    print('❌ Installation failed:', result.stderr)
    print('Try manually: !pip install anthropic')

# Verify import works
try:
    import anthropic
    print(f'✅ anthropic version: {anthropic.__version__}')
except ImportError as e:
    print(f'❌ Import failed: {e}. Please restart runtime and re-run.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 ── Secure API Key Setup
# ═══════════════════════════════════════════════════════════════
import os
from getpass import getpass

print('=' * 55)
print('🔐  ANTHROPIC API KEY SETUP')
print('=' * 55)
print('📌 Get your FREE key at: https://console.anthropic.com')
print('📌 Your key starts with: sk-ant-api03-...')
print('📌 Input is HIDDEN for security (normal to see nothing)')
print('─' * 55)

# Try Colab Secrets first (if user stored it there)
api_key = ''
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    if api_key:
        print('✅ Found API key in Colab Secrets!')
except Exception:
    pass

# If not found in secrets, ask user
if not api_key:
    api_key = getpass('\nPaste your Anthropic API Key: ')

# Validate key format
api_key = api_key.strip()

if not api_key:
    print('\n❌ ERROR: No API key entered. Please re-run this cell.')
elif not api_key.startswith('sk-ant-'):
    print('\n⚠️  WARNING: Key does not start with sk-ant-')
    print('   Make sure you copied the full key correctly.')
    print('   Saving anyway — it will show an error in Cell 3 if wrong.')
    os.environ['ANTHROPIC_API_KEY'] = api_key
else:
    os.environ['ANTHROPIC_API_KEY'] = api_key
    masked = api_key[:12] + '...' + api_key[-4:]
    print(f'\n✅ API Key saved! (masked: {masked})')
    print('✅ Ready! Proceed to Cell 3.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 ── AlphaBot Full Chatbot (MAIN — Run This Last)
# ═══════════════════════════════════════════════════════════════

# ── Imports ─────────────────────────────────────────────────────
import os
import sys
import json
import time
import re
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Dependency Guard ────────────────────────────────────────────
try:
    import anthropic
except ImportError:
    print('❌ anthropic not installed. Please run Cell 1 first, then re-run this cell.')
    raise SystemExit

# ── API Key Guard ───────────────────────────────────────────────
API_KEY = os.environ.get('ANTHROPIC_API_KEY', '').strip()
if not API_KEY:
    print('❌ No API Key found. Please run Cell 2 first, then re-run this cell.')
    raise SystemExit

# ─────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────
BOT_NAME    = 'AlphaBot'
MODEL       = 'claude-sonnet-4-20250514'
MAX_TOKENS  = 1024
MAX_HISTORY = 40   # messages kept in memory
TODAY       = datetime.now().strftime('%A, %B %d, %Y')
NOW_TIME    = datetime.now().strftime('%I:%M %p')

SYSTEM_PROMPT = f"""You are {BOT_NAME}, a friendly, intelligent, and helpful AI assistant.
Rules:
- Be concise but complete (2-5 sentences unless user asks for more)
- Be warm, encouraging, and occasionally use a relevant single emoji
- You remember the FULL conversation — reference it when relevant
- If asked to write code, use proper markdown code blocks with language labels
- If asked math, show step-by-step working
- If asked something you are unsure about, say so honestly
- Never make up facts
- Today is {TODAY}. Current time: {NOW_TIME}.
- You are running inside Google Colab as part of a CodeAlpha internship project"""

# ─────────────────────────────────────────────────────────────────
# ANTHROPIC CLIENT
# ─────────────────────────────────────────────────────────────────
try:
    client = anthropic.Anthropic(api_key=API_KEY)
except Exception as e:
    print(f'❌ Failed to create Anthropic client: {e}')
    raise SystemExit

# ─────────────────────────────────────────────────────────────────
# CONVERSATION STATE
# ─────────────────────────────────────────────────────────────────
conversation_history = []   # [{role, content}, ...]
message_count        = [0]
session_start        = datetime.now()
error_log            = []   # stores errors for debugging

# ─────────────────────────────────────────────────────────────────
# QUICK / RULE-BASED REPLIES (instant, no API cost)
# ─────────────────────────────────────────────────────────────────
def get_quick_reply(text):
    """Check if message matches a quick rule. Returns reply or None."""
    n = text.strip().lower()

    # Greetings
    if re.search(r'\b(hello|hi|hey|howdy|greetings|sup|yo)\b', n):
        return f'Hello there! 👋 I am {BOT_NAME}, your AI assistant. What can I help you with today?'

    # Farewells
    if re.search(r'\b(bye|goodbye|see you|cya|take care|later|quit|exit)\b', n):
        duration = str(datetime.now() - session_start).split('.')[0]
        return f'Goodbye! 👋 We chatted for {duration}. It was great talking with you — come back anytime!'

    # Thanks
    if re.search(r'\b(thanks|thank you|thx|ty|thank u|appreciate)\b', n):
        return 'You are very welcome! 😊 Happy to help — ask me anything else anytime.'

    # Bot identity
    if re.search(r'(who are you|what are you|your name|what is your name)', n):
        return (f'I am {BOT_NAME}, an AI assistant powered by Anthropic Claude. '
                f'I was built for the CodeAlpha Python Internship. '
                f'I can answer questions, write code, do math, have conversations, and much more! 🤖')

    # Capabilities
    if re.search(r'(what can you do|help|capabilities|features|commands)', n):
        return (
            '🤖 Here is what I can do:\n'
            '• 💬 Answer any question\n'
            '• 💻 Write & explain code (Python, JS, etc.)\n'
            '• 🧮 Solve math problems step-by-step\n'
            '• 📖 Summarize text or topics\n'
            '• 🌍 Translate languages\n'
            '• 😂 Tell jokes\n'
            '• 🧠 Remember our full conversation\n'
            '• 📅 Tell you the date/time\n'
            'Just type naturally — I understand you!'
        )

    # Jokes
    if re.search(r'\b(joke|funny|laugh|humor|hilarious)\b', n):
        jokes = [
            'Why do programmers prefer dark mode? Because light attracts bugs! 🐛',
            'Why do Python programmers wear glasses? Because they cannot C! 😂',
            'How many programmers does it take to change a lightbulb? None — that is a hardware problem! 💡',
            'Why did the AI go to school? To improve its learning rate! 🎓',
            'What do you call a sleeping dinosaur? A dino-snore! 🦕',
        ]
        import random
        return random.choice(jokes)

    # Date / Time
    if re.search(r'\b(time|clock)\b', n) and 'date' not in n:
        return f'⏰ Current time: {datetime.now().strftime("%I:%M:%S %p")}'
    if re.search(r'\b(date|today|day|month|year)\b', n):
        return f'📅 Today is {datetime.now().strftime("%A, %B %d, %Y")}'

    # How are you
    if re.search(r"(how are you|how's it going|are you ok|you good)", n):
        return ("I am running perfectly — no bugs detected! 😄 "
                "My circuits are happy and I am ready to help. How about you?")

    # Creator
    if re.search(r'(who (made|built|created|designed) you|your creator|your developer)', n):
        return ('I was built as part of the CodeAlpha Python Programming Internship. '
                'My AI brain is powered by Anthropic Claude. 🧠')

    return None   # no quick match → send to AI


# ─────────────────────────────────────────────────────────────────
# INPUT VALIDATOR
# ─────────────────────────────────────────────────────────────────
def validate_input(text):
    """
    Returns (is_valid: bool, error_message: str | None)
    Catches all bad inputs a user might send.
    """
    if not text or not text.strip():
        return False, 'empty'

    stripped = text.strip()

    if len(stripped) > 4000:
        return False, f'⚠️ Message too long ({len(stripped)} chars). Please keep it under 4000 characters.'

    if len(stripped) < 1:
        return False, 'empty'

    # Check if message is only special characters / gibberish
    if re.match(r'^[^a-zA-Z0-9\u0900-\u097F]+$', stripped) and len(stripped) < 3:
        return False, '⚠️ Please type a proper message.'

    return True, None


# ─────────────────────────────────────────────────────────────────
# AI RESPONSE ENGINE
# ─────────────────────────────────────────────────────────────────
def get_ai_response(user_text):
    """
    Full response pipeline:
    1. Quick rules (free, instant)
    2. Claude API with full history
    3. Specific error messages for every failure type
    """
    # Step 1: Quick reply check
    quick = get_quick_reply(user_text)
    if quick:
        return quick, 'quick'

    # Step 2: Add to history and call API
    conversation_history.append({'role': 'user', 'content': user_text})

    # Trim history to stay within limits (keep most recent)
    history_to_send = conversation_history[-MAX_HISTORY:]

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=SYSTEM_PROMPT,
            messages=history_to_send,
        )
        reply = response.content[0].text
        # Save assistant reply to history
        conversation_history.append({'role': 'assistant', 'content': reply})
        return reply, 'ai'

    # ── Specific error handling ──────────────────────────────────
    except anthropic.AuthenticationError:
        conversation_history.pop()  # remove the user msg we just added
        err = ('❌ Invalid API Key.\n'
               'Fix: Re-run Cell 2 and paste a valid key from\n'
               'https://console.anthropic.com')
        error_log.append({'time': datetime.now().isoformat(), 'error': 'AuthenticationError'})
        return err, 'error'

    except anthropic.PermissionDeniedError:
        conversation_history.pop()
        err = ('❌ Permission Denied.\n'
               'Your API key does not have access to this model.\n'
               'Fix: Check your Anthropic account at console.anthropic.com')
        error_log.append({'time': datetime.now().isoformat(), 'error': 'PermissionDeniedError'})
        return err, 'error'

    except anthropic.RateLimitError:
        conversation_history.pop()
        err = ('⚠️ Rate Limit Reached.\n'
               'You have sent too many requests too fast.\n'
               'Fix: Wait 30-60 seconds, then try again.')
        error_log.append({'time': datetime.now().isoformat(), 'error': 'RateLimitError'})
        return err, 'error'

    except anthropic.APIConnectionError:
        conversation_history.pop()
        err = ('🔌 Connection Error.\n'
               'Cannot reach the Anthropic servers.\n'
               'Fix: Check your internet connection and try again.')
        error_log.append({'time': datetime.now().isoformat(), 'error': 'APIConnectionError'})
        return err, 'error'

    except anthropic.APITimeoutError:
        conversation_history.pop()
        err = ('⏱️ Request Timed Out.\n'
               'The server took too long to respond.\n'
               'Fix: Try again in a moment.')
        error_log.append({'time': datetime.now().isoformat(), 'error': 'APITimeoutError'})
        return err, 'error'

    except anthropic.BadRequestError as e:
        conversation_history.pop()
        err = (f'⚠️ Bad Request: {str(e)[:100]}\n'
               'Your message may contain unsupported content.\n'
               'Fix: Try rephrasing your message.')
        error_log.append({'time': datetime.now().isoformat(), 'error': f'BadRequestError: {e}'})
        return err, 'error'

    except anthropic.InternalServerError:
        conversation_history.pop()
        err = ('🛠️ Anthropic Server Error.\n'
               'Something went wrong on the API side (not your fault).\n'
               'Fix: Wait a moment and try again.')
        error_log.append({'time': datetime.now().isoformat(), 'error': 'InternalServerError'})
        return err, 'error'

    except KeyboardInterrupt:
        conversation_history.pop()
        return '⏹️ Request cancelled.', 'error'

    except Exception as e:
        conversation_history.pop()
        err = (f'⚠️ Unexpected Error: {type(e).__name__}: {str(e)[:150]}\n'
               'Please screenshot this and report it.')
        error_log.append({'time': datetime.now().isoformat(), 'error': f'{type(e).__name__}: {e}'})
        return err, 'error'


# ─────────────────────────────────────────────────────────────────
# CSS — Dark Futuristic Theme
# ─────────────────────────────────────────────────────────────────
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Sora:wght@300;400;600;700&display=swap');

* { box-sizing: border-box; margin: 0; padding: 0; }

.ab-wrap {
  font-family: 'Sora', sans-serif;
  background: linear-gradient(160deg, #080810 0%, #130a28 40%, #080d1a 100%);
  border: 1px solid #2d1f5e;
  border-radius: 22px;
  padding: 22px;
  max-width: 740px;
  margin: 8px auto;
  box-shadow: 0 0 80px rgba(139, 92, 246, 0.12), inset 0 1px 0 rgba(255,255,255,0.04);
}

.ab-header {
  text-align: center;
  padding-bottom: 14px;
  border-bottom: 1px solid rgba(139,92,246,0.2);
  margin-bottom: 16px;
}
.ab-header h2 {
  color: #a78bfa;
  font-size: 1.55rem;
  font-weight: 700;
  letter-spacing: 3px;
  text-transform: uppercase;
  text-shadow: 0 0 20px rgba(167,139,250,0.4);
}
.ab-header p { color: #6b7280; font-size: 0.72rem; margin-top: 5px; letter-spacing: 1.5px; }

.ab-stats {
  display: flex;
  justify-content: space-between;
  font-size: 0.65rem;
  color: #4b5563;
  font-family: 'JetBrains Mono', monospace;
  padding: 0 4px;
  margin-bottom: 12px;
}
.ab-dot {
  display: inline-block; width: 6px; height: 6px;
  background: #22c55e; border-radius: 50%; margin-right: 5px;
  animation: abpulse 2s infinite;
}
.ab-dot.thinking { background: #f59e0b; }
@keyframes abpulse { 0%,100%{opacity:1;} 50%{opacity:0.25;} }

.ab-log {
  background: rgba(0,0,0,0.35);
  border: 1px solid #1f2937;
  border-radius: 16px;
  padding: 16px 14px;
  min-height: 320px;
  max-height: 440px;
  overflow-y: auto;
  display: flex;
  flex-direction: column;
  gap: 14px;
  scroll-behavior: smooth;
}
.ab-log::-webkit-scrollbar { width: 4px; }
.ab-log::-webkit-scrollbar-track { background: transparent; }
.ab-log::-webkit-scrollbar-thumb { background: #3730a3; border-radius: 4px; }

.ab-row { display: flex; align-items: flex-end; gap: 9px; }
.ab-row.user { flex-direction: row-reverse; }

.ab-avatar {
  width: 36px; height: 36px; border-radius: 50%;
  display: flex; align-items: center; justify-content: center;
  font-size: 1.05rem; flex-shrink: 0;
}
.ab-avatar.bot  { background: linear-gradient(135deg, #6d28d9, #a78bfa); box-shadow: 0 0 12px rgba(109,40,217,0.5); }
.ab-avatar.user { background: linear-gradient(135deg, #065f46, #34d399); box-shadow: 0 0 12px rgba(6,95,70,0.5); }

.ab-msg-wrap { display: flex; flex-direction: column; max-width: 80%; }
.ab-row.user .ab-msg-wrap { align-items: flex-end; }

.ab-bubble {
  padding: 10px 15px;
  border-radius: 18px;
  font-size: 0.855rem;
  line-height: 1.65;
  word-break: break-word;
  white-space: pre-wrap;
}
.ab-bubble.bot {
  background: linear-gradient(135deg, #1e1650 0%, #2d1f6e 100%);
  color: #ddd6fe;
  border: 1px solid #4338ca;
  border-bottom-left-radius: 4px;
}
.ab-bubble.user {
  background: linear-gradient(135deg, #052e1c 0%, #064e3b 100%);
  color: #a7f3d0;
  border: 1px solid #047857;
  border-bottom-right-radius: 4px;
}
.ab-bubble.error {
  background: linear-gradient(135deg, #3b0a0a, #450a0a);
  color: #fca5a5;
  border: 1px solid #7f1d1d;
  border-bottom-left-radius: 4px;
  font-family: 'JetBrains Mono', monospace;
  font-size: 0.78rem;
}

.ab-sender {
  font-size: 0.62rem; font-weight: 700;
  letter-spacing: 1.2px; text-transform: uppercase;
  opacity: 0.55; margin-bottom: 4px;
}
.ab-bubble.bot  .ab-sender, .ab-bubble.error .ab-sender { color: #a78bfa; }
.ab-bubble.user .ab-sender { color: #34d399; }

.ab-ts {
  font-size: 0.58rem; color: #374151;
  font-family: 'JetBrains Mono', monospace;
  margin-top: 5px; padding: 0 2px;
}

.ab-thinking {
  display: flex; gap: 5px;
  padding: 12px 16px;
  background: linear-gradient(135deg, #1e1650, #2d1f6e);
  border: 1px solid #4338ca;
  border-radius: 18px; border-bottom-left-radius: 4px;
  width: fit-content;
}
.ab-dot-b {
  width: 7px; height: 7px;
  background: #a78bfa; border-radius: 50%;
  animation: abbounce 1.3s infinite;
}
.ab-dot-b:nth-child(2) { animation-delay: 0.22s; }
.ab-dot-b:nth-child(3) { animation-delay: 0.44s; }
@keyframes abbounce { 0%,80%,100%{transform:translateY(0);opacity:0.35;} 40%{transform:translateY(-7px);opacity:1;} }

.ab-divider {
  border: none; border-top: 1px solid rgba(55,48,163,0.3);
  margin: 4px 0;
}

/* system message */
.ab-system {
  text-align: center;
  font-size: 0.65rem;
  color: #4b5563;
  font-family: 'JetBrains Mono', monospace;
  letter-spacing: 0.5px;
  padding: 2px 0;
}
</style>
"""


# ─────────────────────────────────────────────────────────────────
# HTML BUILDERS
# ─────────────────────────────────────────────────────────────────
def esc(text):
    """Escape HTML special characters."""
    return (text
            .replace('&', '&amp;')
            .replace('<', '&lt;')
            .replace('>', '&gt;')
            .replace('"', '&quot;'))

def make_bot_bubble(text, is_error=False):
    ts    = datetime.now().strftime('%I:%M:%S %p')
    cls   = 'error' if is_error else 'bot'
    icon  = '⚠️' if is_error else '🤖'
    safe  = esc(text)
    return f"""
<div class='ab-row bot'>
  <div class='ab-avatar bot'>{icon}</div>
  <div class='ab-msg-wrap'>
    <div class='ab-bubble {cls}'>
      <div class='ab-sender'>{BOT_NAME}</div>
      {safe}
    </div>
    <div class='ab-ts'>{ts}</div>
  </div>
</div>"""

def make_user_bubble(text):
    ts   = datetime.now().strftime('%I:%M:%S %p')
    safe = esc(text)
    return f"""
<div class='ab-row user'>
  <div class='ab-avatar user'>🧑</div>
  <div class='ab-msg-wrap'>
    <div class='ab-bubble user'>
      <div class='ab-sender'>You</div>
      {safe}
    </div>
    <div class='ab-ts'>{ts}</div>
  </div>
</div>"""

def make_system_msg(text):
    return f"<div class='ab-system'>── {esc(text)} ──</div>"

THINKING_HTML = """
<div class='ab-row bot' id='ab-thinking'>
  <div class='ab-avatar bot'>🤖</div>
  <div class='ab-thinking'>
    <div class='ab-dot-b'></div>
    <div class='ab-dot-b'></div>
    <div class='ab-dot-b'></div>
  </div>
</div>"""

SCROLL_JS = "<script>setTimeout(()=>{var e=document.querySelector('.ab-log');if(e)e.scrollTop=e.scrollHeight;},80);</script>"


# ─────────────────────────────────────────────────────────────────
# RENDER ENGINE
# ─────────────────────────────────────────────────────────────────
chat_log_html = []   # list of HTML strings for each bubble

def render(thinking=False, status_override=None):
    mc   = message_count[0]
    hc   = len(conversation_history)
    ec   = len(error_log)
    dur  = str(datetime.now() - session_start).split('.')[0]

    if status_override:
        status_text = status_override
        dot_cls = 'ab-dot thinking'
    elif thinking:
        status_text = 'THINKING...'
        dot_cls = 'ab-dot thinking'
    else:
        status_text = 'ONLINE'
        dot_cls = 'ab-dot'

    error_badge = f' | ⚠️ Errors: {ec}' if ec > 0 else ''

    stats = f"""
<div class='ab-stats'>
  <span><span class='{dot_cls}'></span>{status_text} — {BOT_NAME}</span>
  <span>💬 {mc} msgs | 🧠 {hc} in memory{error_badge}</span>
  <span>⏱ {dur}</span>
</div>"""

    bubbles = '\n'.join(chat_log_html)
    if thinking:
        bubbles += THINKING_HTML

    full = f"""
{CSS}
<div class='ab-wrap'>
  <div class='ab-header'>
    <h2>🤖 {BOT_NAME}</h2>
    <p>CODEALPHA INTERNSHIP — AI CHATBOT — GOOGLE COLAB</p>
  </div>
  {stats}
  <div class='ab-log'>{bubbles}</div>
</div>
{SCROLL_JS}"""

    with output_area:
        clear_output(wait=True)
        display(HTML(full))


# ─────────────────────────────────────────────────────────────────
# WIDGETS
# ─────────────────────────────────────────────────────────────────
output_area = widgets.Output()

text_input = widgets.Text(
    placeholder='💬  Type your message here and press Enter...',
    layout=widgets.Layout(width='72%', height='42px'),
)
send_btn = widgets.Button(
    description='Send ➤',
    tooltip='Send message (or press Enter)',
    layout=widgets.Layout(width='13%', height='42px'),
)
clear_btn = widgets.Button(
    description='🗑 Clear',
    tooltip='Clear chat and reset memory',
    layout=widgets.Layout(width='10%', height='42px'),
)
debug_btn = widgets.Button(
    description='🐛 Debug',
    tooltip='Show error log',
    layout=widgets.Layout(width='10%', height='42px'),
)
send_btn.style.button_color  = '#6d28d9'
clear_btn.style.button_color = '#374151'
debug_btn.style.button_color = '#78350f'

input_row = widgets.HBox(
    [text_input, send_btn, clear_btn, debug_btn],
    layout=widgets.Layout(gap='5px', margin='8px 0 0 0'),
)


# ─────────────────────────────────────────────────────────────────
# EVENT HANDLERS
# ─────────────────────────────────────────────────────────────────
def on_send(event=None):
    raw_text = text_input.value

    # Validate input
    valid, err_msg = validate_input(raw_text)
    if not valid:
        if err_msg and err_msg != 'empty':
            chat_log_html.append(make_bot_bubble(err_msg, is_error=True))
            render()
        text_input.value = ''
        return

    user_text = raw_text.strip()
    text_input.value = ''

    # Disable UI during processing
    send_btn.disabled = True
    text_input.disabled = True
    message_count[0] += 1

    # Show user message + thinking indicator
    chat_log_html.append(make_user_bubble(user_text))
    render(thinking=True)

    # Get response
    try:
        reply, reply_type = get_ai_response(user_text)
        is_err = (reply_type == 'error')
        chat_log_html.append(make_bot_bubble(reply, is_error=is_err))
    except Exception as e:
        chat_log_html.append(make_bot_bubble(
            f'⚠️ Critical failure: {type(e).__name__}: {e}', is_error=True
        ))

    # Re-enable UI
    send_btn.disabled = False
    text_input.disabled = False
    render()


def on_clear(event=None):
    chat_log_html.clear()
    conversation_history.clear()
    error_log.clear()
    message_count[0] = 0
    chat_log_html.append(make_system_msg('Chat cleared — memory reset'))
    chat_log_html.append(make_bot_bubble(
        'Chat cleared! 🧹 Fresh start. How can I help you?'
    ))
    render()


def on_debug(event=None):
    if not error_log:
        chat_log_html.append(make_bot_bubble('✅ No errors recorded in this session. All good!'))
    else:
        lines = [f'🐛 Error Log ({len(error_log)} entries):']
        for i, e in enumerate(error_log, 1):
            lines.append(f'{i}. [{e["time"]}] {e["error"]}')
        chat_log_html.append(make_bot_bubble('\n'.join(lines), is_error=True))
    render()


send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
debug_btn.on_click(on_debug)
text_input.on_submit(on_send)


# ─────────────────────────────────────────────────────────────────
# LAUNCH
# ─────────────────────────────────────────────────────────────────
chat_log_html.append(make_system_msg(f'Session started — {TODAY} at {NOW_TIME}'))
chat_log_html.append(make_bot_bubble(
    f'Hello! 👋 I am {BOT_NAME}, your AI assistant powered by Claude.\n\n'
    f'I remember our full conversation throughout this session.\n'
    f'You can ask me anything — questions, code, math, jokes, or just chat!\n\n'
    f'Type "help" to see everything I can do. What is on your mind?'
))

display(output_area)
display(input_row)
render()
print('✅ AlphaBot is live! Type in the box above and press Enter or click Send.')
print('   Buttons: [Send ➤] send message | [🗑 Clear] reset chat | [🐛 Debug] view errors')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 (Optional) ── Export & Download Conversation Log
# ═══════════════════════════════════════════════════════════════
from datetime import datetime

if not conversation_history:
    print('⚠️ No conversation to export yet. Chat first, then run this cell.')
else:
    filename = f"AlphaBot_Chat_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

    with open(filename, 'w', encoding='utf-8') as f:
        f.write('═' * 60 + '\n')
        f.write('  AlphaBot — Conversation Log\n')
        f.write(f'  Exported: {datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")}\n')
        f.write(f'  Total messages: {len(conversation_history)}\n')
        f.write('═' * 60 + '\n\n')

        for msg in conversation_history:
            role = '🧑 You' if msg['role'] == 'user' else f'🤖 {BOT_NAME}'
            f.write(f'{role}:\n')
            f.write(msg['content'])
            f.write('\n\n' + '─' * 40 + '\n\n')

    print(f'✅ Saved: {filename}')

    try:
        from google.colab import files
        files.download(filename)
        print('✅ Download started!')
    except Exception as e:
        print(f'ℹ️ Auto-download unavailable ({e}). Find your file in the Colab file browser (left sidebar).')